In [2]:
import kagglehub
import pandas as pd
import json
from pathlib import Path


# Download latest version
path = kagglehub.dataset_download("camilonunez/magic-the-gathering-top8-some-decks-and-events")

print("Path to dataset files:", path)

C:\Users\nagvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\nagvi\.cache\kagglehub\datasets\camilonunez\magic-the-gathering-top8-some-decks-and-events\versions\3


In [ ]:
RUNTIME_WARNINGS = False

df = pd.read_csv(path + "/df_events_v2.csv")
#print(df.columns)

def open_decks(event_id):
    #ID alapján - ha van, - visszaküldi az adott eventhez tartozó deckek dataFrame-ét: player_name, result, main_deck, sideboard
    idx = df[df['event__id'] == event_id].index.tolist()
    if len(idx) == 0:
        if RUNTIME_WARNINGS:
            print("Could not find an event with ID:", event_id)
        return pd.DataFrame(columns = ["player", "result", "main_deck", "sideboard"])
    elif len(idx) > 1:
        raise RuntimeError("More than one event with matching ID")
    else:
        idx = idx[0]
    rows = []    
    folder = Path(path + "/events/" + str(idx + 2) + "/players_decks")
    #Ellenőrizzük, hogy vannak-e decklisták az adott eventhez
    json_files = list(folder.glob("*.json"))
    if len(json_files) == 0:
        if RUNTIME_WARNINGS:
            print("No deck data for this event:", idx+2)
        return pd.DataFrame(columns = ["player", "result", "main_deck", "sideboard"])
    #A játékos deckek json fájlok, ezek beolvasása
    for file in folder.glob("*.json"):
        try:
            with open(file, "r", encoding="utf-8") as f:
                content = f.read().strip()
                if not content:
                    if RUNTIME_WARNINGS:
                        print(f"Üres fájl: {file}")
                    continue
                data = json.loads(content)
                rows.append(data)
        except json.JSONDecodeError:
            if RUNTIME_WARNINGS:
                print(f"Hibás JSON: {file}")
            continue
    return_df = pd.DataFrame(rows)
    # Ha nincs adat
    if return_df.empty:
        return return_df
    #Csak ha léteznek az oszlopok
    if "main_deck" in return_df.columns:
        return_df["main_deck"] = return_df["main_deck"].apply(lambda d: {k: int(v) for k, v in dict(d).items()})
    if "sideboard" in return_df.columns:
        return_df["sideboard"] = return_df["sideboard"].apply(lambda d: {k: int(v) for k, v in dict(d).items()})

    return return_df
    
#Pl.
open_decks(29436)

NameError: name 'pd' is not defined

In [ ]:
#Ez a cella 10+ percig fut
all_decks = []

for row in df.itertuples():
    if row.event_format != 'ST':
        continue
    current_frame = open_decks(row.event__id)
    if current_frame.empty:
        continue
    all_decks.append(current_frame)
all_decks_df = pd.concat(all_decks, ignore_index=True)

card_frequencies = {}
for j, r in all_decks_df.iterrows():
    for card in r["main_deck"].keys():
        if card not in card_frequencies.keys():
            card_frequencies.update({card: r["main_deck"][card]})
        else:
            card_frequencies[card] += r["main_deck"][card]

#Lementjük a releváns eredményeket, hogy ne kelljen a lassú algoritmusokat mindig újra futtatni
all_decks_df.to_pickle('all_decks_df.pickle')

with open("card_frequencies", "w", encoding="utf-8") as f:
    json.dump(card_frequencies, f, ensure_ascii=False, indent=2)

#Összességében egy 'all_deck_df.pickle' fájlt, és egy 'card_frequencies.json' fájlt csinálunk